In [1]:

!pip install -q -U torch transformers peft datasets bitsandbytes trl
!pip install tensorrt_llm -U --pre --extra-index-url https://pypi.nvidia.com
!git clone https://github.com/NVIDIA/TensorRT-LLM.git

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 22.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorrt-llm 1.2.0rc5 requires datasets==3.1.0, but you have datasets 4.4.1 which is incompatible.
tensorrt-llm 1.2.0rc5 requires torch<=2.9.0,>=2.9.0a0, but you have torch 2.9.1 which is incompatible.
tensorrt-llm 1.2.0rc5 requires transformers==4.56.0, but you have transformers 4.57.3 which is incompatible.
tensorrt-llm 1.2.0rc5 requires triton==3.5.0; platform_machine == "x86_64", but you have triton 3.5.1 which is incompatible.
torchaudio 2.9.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
torchvision 0.24.0+cu126 requires torch==2.9.0, but you have torch 2.9.1 which is incompatible.
Looking in indexes: https://pypi.org/sim

In [2]:
from huggingface_hub import login
login(new_session=True)

In [3]:
import torch, gc

try:
    del model
    del trainer
    del tokenizer
except:
    pass

gc.collect()

torch.cuda.empty_cache()

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

dataset = load_dataset("json", data_files="robot_dataset.json", split="train")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

def preprocess_function(example):
    text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>"

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_function, batched=False, remove_columns=dataset.column_names)
print("전처리 완료!")

peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=SFTConfig(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        logging_steps=10,
        learning_rate=2e-4,

        fp16=True,
        bf16=False,

        optim="paged_adamw_32bit",
        weight_decay=0.001,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type="constant",
    ),
)

print("학습을 시작합니다...")
trainer.train()

new_model_name = "trained_adapter"
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print("학습 완료! 어댑터 저장됨.")

Generating train split: 0 examples [00:00, ? examples/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: juun03 (juun03-yonsei-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# 1. 모델 ID
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

# 2. 데이터셋 로드
dataset = load_dataset("json", data_files="robot_dataset.json", split="train")

# 3. BitsAndBytes 설정 (4비트 + FP16 연산)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # 연산 타입 FP16
    bnb_4bit_use_double_quant=True,
    llm_int8_enable_fp32_cpu_offload=True
)

# 4. 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16  # 로드 시 타입 FP16
)

# [수술 단계 1] 모델 설정 강제 변경
model.config.torch_dtype = torch.float16
model.config.use_cache = False
model.config.pretraining_tp = 1

# [수술 단계 2] 학습 안정성을 위한 전처리 (LayerNorm 등을 fp32로 변환)
# SFTTrainer 내부에서 하던 걸 밖으로 꺼내서 우리가 직접 제어합니다.
model = prepare_model_for_kbit_training(model)

# 5. 토크나이저 설정
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 6. 데이터 전처리 함수 (채팅 포맷 + 토크나이징)
def preprocess_function(example):
    text = f"<|begin_of_text|><|start_header_id|>user<|end_header_id|>\n\n{example['instruction']}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{example['output']}<|eot_id|>"
    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=512,
        padding="max_length",
        return_tensors=None
    )
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

print("데이터셋 전처리 중...")
dataset = dataset.map(preprocess_function, batched=False, remove_columns=dataset.column_names)
print("전처리 완료!")

# 7. LoRA 어댑터 수동 부착
# SFTTrainer에 peft_config를 넘기지 않고, 우리가 직접 모델에 붙여서 타입을 확정합니다.
peft_config = LoraConfig(
    r=64,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# [수술 단계 3 - 핵심] 좀비처럼 살아남은 BFloat16 탐색 및 사살
# 모델의 모든 파라미터를 뒤져서 bfloat16이 있으면 float16으로 강제 형변환합니다.
print("BFloat16 잔재 제거 중...")
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        print(f"변환됨: {name}")
        param.data = param.data.to(torch.float16)

# LoRA 레이어들도 확실하게 float16으로 맞춥니다.
for name, module in model.named_modules():
    if 'lora' in name or 'adapter' in name:
        module.to(torch.float16)
print("타입 정리 완료.")

# 8. 학습 설정
# model이 이미 PEFT 모델이므로 peft_config=None으로 설정합니다.
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=None,  # [중요] 위에서 이미 붙였으므로 None
    args=SFTConfig(
        output_dir="./results",
        num_train_epochs=1,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        logging_steps=10,
        learning_rate=2e-4,

        # FP16 강제, BF16 차단
        fp16=True,
        bf16=False,

        optim="paged_adamw_32bit",
        weight_decay=0.001,
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        group_by_length=True,
        lr_scheduler_type="constant",
    ),
)

print("학습을 시작합니다...")
trainer.train()

# 모델 저장
trainer.model.save_pretrained("./trained_adapter")
tokenizer.save_pretrained("./trained_adapter")
print("학습 완료! 어댑터 저장됨.")

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.96 GiB. GPU 0 has a total capacity of 14.74 GiB of which 1.25 GiB is free. Process 101462 has 13.49 GiB memory in use. Of the allocated memory 12.27 GiB is allocated by PyTorch, and 1.09 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)